# 02_preprocessing

Missing-value handling, imputation, train/test split and feature scaling.

---

*DiaPredict AI — Explainable Diabetes Risk Prediction System.* *Educational / research prototype, not a medical diagnosis.*

## Objectives
- Demonstrate the full preprocessing pipeline
- Convert physiologically impossible zeros to missing values
- Impute with the training median and standardise features
- Show how the transforms avoid data leakage

In [ ]:
import sys
from pathlib import Path

# Make the project root importable no matter where the notebook runs.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "backend").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
print("Root:", PROJECT_ROOT)


### 1. Cleaning — impossible zeros → NaN

In [ ]:
import pandas as pd
from backend.app.preprocessing import clean_dataset, load_raw_dataset

raw = load_raw_dataset()
cleaned = clean_dataset(raw)
print(raw[["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]].min())
print()
print("Rows with at least one missing value after cleaning:", cleaned.isnull().any(axis=1).sum())

### 2. Missing-value imputation (median)

In [ ]:
from backend.app.preprocessing import impute_median

imputed, medians = impute_median(cleaned)
print("Median used for imputation / feature:")
for k, v in medians.items():
    print(f"  {k:26s} {v:.2f}")
print("\nRemaining NaNs:", imputed.isnull().sum().sum())

### 3. Train / test split + scaling

In [ ]:
from backend.app.preprocessing import make_train_test

X_train, y_train, X_test, y_test, scaler, medians = make_train_test(raw)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train positives:", y_train.sum(), "/", len(y_train))
print("Test positives: ", y_test.sum(), "/", len(y_test))
print("Mean (train, scaled):", np.round(X_train.mean(axis=0), 2))
print("Std  (train, scaled):", np.round(X_train.std(axis=0), 2))

### 4. Standardisation visualisation

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(raw["Glucose"], bins=30, color="#60a5fa", edgecolor="white")
plt.title("Raw Glucose")
plt.subplot(1, 2, 2)
plt.hist(X_train[:, 1], bins=30, color="#f87171", edgecolor="white")
plt.title("Scaled Glucose (train split)")
plt.tight_layout()
plt.show()

### 5. Why the scaler/medians must be fitted on training only

The `StandardScaler` and the imputation medians are computed **only on the training split** and reused verbatim on the test split / at inference time. This prevents **data leakage** — information from the test set must never influence how training data is scaled, otherwise the test metrics are optimistically biased.

## Summary

Preprocessing closes the loop from a messy raw dataset to a clean, standardised feature matrix ready for model training. The same transforms are replayed on a single patient record in the backend (`backend/app/preprocessing.py`, `prepare_feature_vector` in `services.py`).